### 为什么需要工具调用？

- 解决LLM的局限性：大语言模型存在“幻觉”问题（生成虚假信息），且无法直接访问实时数据或执行具体操作
- 扩展能力边界：通过工具集成，Agent可以获得计算、API访问、数据库查询等实际能力
- 模块化设计：将不同功能封装为独立工具，便于维护和复用。

### LangChain工具开发核心组件

`@tool`装饰器
- 用于将普通Python函数转换为LangChain工具
- 自动生成JSON Schema，描述工具的输入参数和功能
- 示例：
```python
from langchain.tools import tool

@tool
def add_numbers(a: int, b: int) -> int:
    """将两个数字相加"""
    return a + b
```

### JSON Schema规范

- 工具的输入参数需要明确类型（如`str`, `int`）
- 文档字符串（docstring）描述工具功能，Agent依靠此判断是否调用
- JSON Schema示例（自动生成）：
```json
{
  "name": "add_numbers",
  "description": "将两个数字相加",
  "parameters": {
    "type": "object",
    "properties": {
      "a": {"type": "integer"},
      "b": {"type": "integer"}
    },
    "required": ["a", "b"]
  }
}
```

In [1]:
from langchain.tools import tool

In [2]:
@tool
def multiply(a:int,  b:int) -> int:
    """两个数的乘积"""
    return a * b

print(multiply.name)
print(multiply.description)
print(multiply.args)
print(multiply.args_schema.model_json_schema())

multiply
两个数的乘积
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
{'description': '两个数的乘积', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


In [3]:
# 获取城市编码
import pandas as pd

city_df = pd.read_csv("city.csv")

def get_city_code(city_name: str) -> int:
    """
    获取城市编码
    :param city_name: 城市名称
    :return: 返回城市编码
    """
    # 优先匹配区县
    match = city_df[city_df['district'] == city_name]
    if not match.empty:
        return match.iloc[0]['areacode/城市ID']
    # 匹配城市
    match = city_df[city_df['city'] == city_name]
    if not match.empty:
        return match.iloc[0]['areacode/城市ID']
    # 模糊匹配
    match = city_df[city_df['city'].str.contains(city_name, na=False)]
    if not match.empty:
        return match.iloc[0]['areacode/城市ID']
    # 返回北京
    return 101010100

# city_df.head()

get_city_code('丰台')

101010900

In [8]:
# 天气查询工具封装

import requests
import os
from langchain.tools import tool

@tool
def get_weather(city:str) -> str:
    """
    调用实时天气API，返回温度及天气状况
    :param city: 城市编码
    :return: 返回天气状况
    """
    url = "https://eolink.o.apispace.com/456456/weather/v001/now"
    city_code = get_city_code(city)
    payload = {"areacode" : city_code}
    headers = {
        "X-APISpace-Token": os.getenv('API_WEATHER_TOKEN')
    }
    response=requests.request("GET", url, params=payload, headers=headers)
    # 将结果转换为json数据
    data = response.json()
    temp = data.get('result').get('realtime').get('temp')
    tq = data.get('result').get('realtime').get('text')
    return f'天气：{tq}，温度：{temp}℃'

# get_weather('北京')


### LangChain工具调用机制

- LangChain通过`@tool`装饰器将Python函数转换为Agent可调用的工具
- 工具定义遵循JSON Schema规范，确保结构化输入与输出
- Agent通过提示词（Prompt）决定何时调用哪个工具
- 流程：
  1、用户数 -> Agent解析意图
  2、匹配工具 -> 执行函数
  3、返回结果 -> 整合到Agent响应



### 调用工具

In [5]:
import os
from langchain_community.chat_models import ChatZhipuAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
import warnings

warnings.filterwarnings("ignore", message=".*HMAC key is 16 bytes long.*")

key = os.getenv('ZHIPU_API_KEY')
os.environ["ZHIPUAI_API_KEY"] = key

chat = ChatZhipuAI(
    model="glm-4",
    temperature=0.5,
)

In [9]:
from langchain.agents import create_react_agent, AgentExecutor # create_react_agent创建智能体，AgentExecutor执行智能体
from langchain import hub # langChain提供的提示提模板

# 创建工具对象
tools = [get_weather]
# 获取提示词
prompt = hub.pull("hwchase17/react")

# 创建智能体
agent = create_react_agent(llm=chat, tools=tools, prompt=prompt)
# 创建AgentExecutor，运行智能体
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True # 处理解析错误
)
# 调用智能体
result = agent_executor.invoke({'input': "今天北京天气如何？"})
print(result)

E:\python\langchain_study\venv\Lib\site-packages\langsmith\client.py:256: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  manifest: dict | list, *, depth: int = 0, max_depth: int = 10, max_width: int = 50




> Entering new AgentExecutor chain...
用户想了解北京今天的天气情况。我需要使用 get_weather 函数来获取北京的天气信息。根据函数描述，我需要传入一个城市编码作为参数。通常，"北京"的城市编码可能是 "beijing" 或 "Beijing"，我先尝试使用 "beijing"。
Action: get_weather
Action Input: beijing
Observ天气：晴，温度：17.9℃I now know the final answer
Final Answer: 今天北京的天气是晴天，温度为17.9℃。

> Finished chain.
{'input': '今天北京天气如何？', 'output': '今天北京的天气是晴天，温度为17.9℃。'}


In [13]:
import re

@tool
def multiply(input_str:str) -> int:
    """两个数的乘积
    args:
        input_str: 输入字符串，格式为a*b
    return: int
    """
    match = re.findall(r'\d+', input_str)
    if len(match) == 2:
        a,b = map(int, match) # 将列表中的元素依次传递给int，转换类型
    return a * b

In [14]:
from langchain.agents import create_react_agent, AgentExecutor # create_react_agent创建智能体，AgentExecutor执行智能体
from langchain import hub # langChain提供的提示提模板

# 创建工具对象
tools = [get_weather, multiply]
# 获取提示词
prompt = hub.pull("hwchase17/react")

# 创建智能体
agent = create_react_agent(llm=chat, tools=tools, prompt=prompt)
# 创建AgentExecutor，运行智能体
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True # 处理解析错误
)
# 调用智能体
result = agent_executor.invoke({'input': "1x2等于几？"})
print(result)

E:\python\langchain_study\venv\Lib\site-packages\langsmith\client.py:256: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  manifest: dict | list, *, depth: int = 0, max_depth: int = 10, max_width: int = 50




> Entering new AgentExecutor chain...
用户问的是"1x2等于几？"。这是一个数学乘法问题，我应该使用multiply函数来计算。

Action: multiply
Action Input: 1*2
Observ2I now know the final answer
Final Answer: 2

> Finished chain.
{'input': '1x2等于几？', 'output': '2'}


In [15]:
test_inputs = [
    "今天北京天气怎么样？",
    "2x3等于几？",
    "帮我修修电脑"
]

for question in test_inputs:
    resp = agent_executor.invoke({"input": question})
    print(resp)



> Entering new AgentExecutor chain...
用户想了解今天北京的天气情况。我需要使用 get_weather 函数来获取北京的天气信息。根据函数定义，get_weather 需要一个城市编码作为参数。虽然用户说的是"北京"，但函数需要的是城市编码。通常，北京的城市编码可能是 "beijing" 或 "101010100"（这是北京在天气API中常用的编码）。我将尝试使用 "beijing" 作为城市编码。
Action: get_weather
Action Input: beijing
Observ天气：晴，温度：17.9℃我已经成功获取到了北京的天气信息。根据API返回的结果，今天北京天气晴朗，温度为17.9摄氏度。

Thought: I now know the final answer
Final Answer: 今天北京天气晴朗，温度为17.9℃。

> Finished chain.
{'input': '今天北京天气怎么样？', 'output': '今天北京天气晴朗，温度为17.9℃。'}


> Entering new AgentExecutor chain...
用户问的是2x3等于几，这是一个数学计算问题，需要计算2和3的乘积。我应该使用multiply工具来计算这个结果。
Action: multiply
Action Input: 2*3
Observ6I now know the final answer
Final Answer: 6

> Finished chain.
{'input': '2x3等于几？', 'output': '6'}


> Entering new AgentExecutor chain...
用户的问题是"帮我修修电脑"。我需要分析这个问题，看看是否可以使用可用的工具(get_weather, multiply)来解决。

1. get_weather: 用于获取特定城市的天气信息。
2. multiply: 用于计算两个数的乘积。

这两个工具都与修理电脑无关。修理电脑是一个复杂的技术问题，通常需要专业的技术知识、硬件检查、软件诊断等，而不仅仅是获取天气信息或数学计算。

由于没有相关的工具可以帮助修理电脑，我应该告诉用户我无法提供具体的修理服务，但可以给出一些一般